In [4]:
import os
import json
import re

#Datset100!

def specific_split_json_file(dataset_name):
    dataset_folder = os.path.join("/home/sastocke/nnUNet/nnUNet_raw", dataset_name, "imagesTr")
    
    # FIX: Properly remove the _000X.nii.gz suffix for all 4 contrasts
    all_files = []
    for f in os.listdir(dataset_folder):
        if f.endswith(".nii.gz"):
            # Remove .nii.gz extension and the _000X channel suffix
            # This regex removes _0000, _0001, _0002, or _0003 at the end
            base = re.sub(r'_000[0-3]\.nii\.gz$', '', f)
            all_files.append(base)
    
    # Remove duplicates since all 4 channels map to the same case
    all_files = list(set(all_files))
    
    # Define all volunteers: 6 to 52 (47 volunteers total)
    all_volunteers = list(range(6, 53))
    
    # Split into 6 groups
    validation_groups = [
        list(range(6, 15)),    # Fold 0: [6, 7, 8, 9, 10, 11, 12, 13, 14] - 9 volunteers
        list(range(15, 23)),   # Fold 1: [15, 16, 17, 18, 19, 20, 21, 22] - 8 volunteers
        list(range(23, 31)),   # Fold 2: [23, 24, 25, 26, 27, 28, 29, 30] - 8 volunteers
        list(range(31, 39)),   # Fold 3: [31, 32, 33, 34, 35, 36, 37, 38] - 8 volunteers
        list(range(39, 46)),   # Fold 4: [39, 40, 41, 42, 43, 44, 45] - 7 volunteers
    ]
    
    # Test set: volunteers 46-52 (7 volunteers) - NEVER used in training or validation
    test_volunteers = list(range(46, 53))
    
    # Volunteers available for training/validation (exclude test set)
    train_val_volunteers = [v for v in all_volunteers if v not in test_volunteers]
    
    splits = []
    for fold_idx, val_volunteers in enumerate(validation_groups):
        # Training set: all train_val volunteers except those in validation for this fold
        train_volunteers = [v for v in train_val_volunteers if v not in val_volunteers]
        
        # Format volunteer numbers as 2-digit strings to match filenames
        train_volunteers_str = [f"{v:02d}" if v < 100 else str(v) for v in train_volunteers]
        val_volunteers_str = [f"{v:02d}" if v < 100 else str(v) for v in val_volunteers]
        
        # Filter files by volunteer number using formatted strings
        train_ids = [f for f in all_files if any(f"Volunteer_{v}_" in f for v in train_volunteers_str)]
        val_ids = [f for f in all_files if any(f"Volunteer_{v}_" in f for v in val_volunteers_str)]
        
        # Ensure filenames are clean
        train_ids = [f.replace("__", "_") for f in train_ids]
        val_ids = [f.replace("__", "_") for f in val_ids]
        
        # Verification: Check for data leakage
        train_vols_found = set()
        val_vols_found = set()
        
        for f in train_ids:
            for v in train_volunteers_str:
                if f"Volunteer_{v}_" in f:
                    train_vols_found.add(int(v))
        
        for f in val_ids:
            for v in val_volunteers_str:
                if f"Volunteer_{v}_" in f:
                    val_vols_found.add(int(v))
        
        # Check for overlap
        overlap = train_vols_found.intersection(val_vols_found)
        if overlap:
            print(f"⚠️  WARNING: Fold {fold_idx} has DATA LEAKAGE! Overlapping volunteers: {sorted(overlap)}")
        else:
            print(f"✓ Fold {fold_idx}: No data leakage detected")
        
        splits.append({"train": train_ids, "val": val_ids})
        
        print(f"  {len(train_ids)} train samples, {len(val_ids)} val samples")
        print(f"  Train volunteers: {sorted(train_vols_found)}")
        print(f"  Val volunteers: {sorted(val_vols_found)}")

    # Save the splits_final.json file
    output_path = os.path.join("/home/sastocke/nnUNet/nnUNet_preprocessed", dataset_name, "splits_final.json")
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    with open(output_path, "w") as f:
        json.dump(splits, f, indent=4)

    print(f"\n{'='*60}")
    print(f"TEST SET (HELD OUT): Volunteers {test_volunteers}")
    print(f"  These {len(test_volunteers)} volunteers are NEVER used in training or validation!")
    print(f"{'='*60}")
    print(f"\nSplits saved to {output_path}")


dataset_names = ['Dataset101_HannumDirVsAveragesLV']

for dataset_name in dataset_names:
    specific_split_json_file(dataset_name)

✓ Fold 0: No data leakage detected
  151 train samples, 53 val samples
  Train volunteers: [15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45]
  Val volunteers: [6, 7, 8, 9, 10, 11, 12, 13, 14]
✓ Fold 1: No data leakage detected
  163 train samples, 41 val samples
  Train volunteers: [6, 7, 8, 9, 10, 11, 12, 13, 14, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45]
  Val volunteers: [15, 16, 17, 18, 19, 20, 21, 22]
✓ Fold 2: No data leakage detected
  158 train samples, 46 val samples
  Train volunteers: [6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45]
  Val volunteers: [23, 24, 25, 26, 27, 28, 29, 30]
✓ Fold 3: No data leakage detected
  171 train samples, 33 val samples
  Train volunteers: [6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 39, 40, 41,

In [2]:
import json

# Path to splits_final.json
split_file_path = "/home/sastocke/nnUNet/nnUNet_preprocessed/Dataset061_DiscoandDirvsAvgHannumIPs/splits_final.json"

# Load and inspect the splits
with open(split_file_path, "r") as f:
    splits = json.load(f)

print(f"Number of splits: {len(splits)}")

# Optionally, print details for each split
for idx, split in enumerate(splits):
    print(f"Split {idx + 1}:")
    print(f"  Train: {len(split['train'])} samples")
    print(f"  Val: {len(split['val'])} samples")


Number of splits: 5
Split 1:
  Train: 2976 samples
  Val: 92 samples
Split 2:
  Train: 3056 samples
  Val: 80 samples
Split 3:
  Train: 3104 samples
  Val: 64 samples
Split 4:
  Train: 3008 samples
  Val: 88 samples
Split 5:
  Train: 3136 samples
  Val: 52 samples


In [24]:
import json

# Paths to split files
old_split_path = "/home/sastocke/nnUNet/nnUNet_preprocessed/Dataset050_DataAugAllSpecifcNormLVOnly/old_splits_final.json"  # Replace with the path of your 11-split file
new_split_path = "/home/sastocke/nnUNet/nnUNet_preprocessed/Dataset050_DataAugAllSpecifcNormLVOnly/splits_final.json"

# Load and print splits
with open(old_split_path, "r") as f:
    old_splits = json.load(f)

with open(new_split_path, "r") as f:
    new_splits = json.load(f)

print(f"Old splits: {len(old_splits)}")
for i, split in enumerate(old_splits):
    print(f"Old Split {i + 1}: Train {len(split['train'])}, Val {len(split['val'])}")

print("\nNew splits: {len(new_splits)}")
for i, split in enumerate(new_splits):
    print(f"New Split {i + 1}: Train {len(split['train'])}, Val {len(split['val'])}")


Old splits: 11
Old Split 1: Train 2592, Val 368
Old Split 2: Train 2640, Val 320
Old Split 3: Train 2704, Val 256
Old Split 4: Train 2608, Val 352
Old Split 5: Train 2752, Val 208
Old Split 6: Train 2736, Val 224
Old Split 7: Train 2720, Val 240
Old Split 8: Train 2704, Val 256
Old Split 9: Train 2752, Val 208
Old Split 10: Train 2704, Val 256
Old Split 11: Train 2688, Val 272

New splits: {len(new_splits)}
New Split 1: Train 2592, Val 368
New Split 2: Train 2640, Val 320
New Split 3: Train 2704, Val 256
New Split 4: Train 2608, Val 352
New Split 5: Train 2752, Val 208


In [5]:
import os
import json
import re


def matches_volunteer(filename, volunteer_id):
    """
    Match Volunteer_6_ against Volunteer_6_ OR Volunteer_06_ OR Volunteer_006_
    """
    return re.search(rf'Volunteer_0*{volunteer_id}_', filename) is not None


def specific_split_json_file(dataset_name):
    dataset_folder = os.path.join(
        "/home/sastocke/nnUNet/nnUNet_raw", dataset_name, "imagesTr"
    )

    # Remove channel suffixes (_000X.nii.gz)
    all_files = []
    for f in os.listdir(dataset_folder):
        if f.endswith(".nii.gz"):
            base = re.sub(r'_000[0-3]\.nii\.gz$', '', f)
            all_files.append(base)

    all_files = sorted(set(all_files))
    print(f"Sample filenames: {all_files[:3]}")

    # Extract volunteer IDs (handles zero-padding correctly)
    volunteer_ids = set()
    for f in all_files:
        m = re.search(r'Volunteer_0*(\d+)_', f)
        if m:
            volunteer_ids.add(int(m.group(1)))

    all_volunteers = sorted(volunteer_ids)
    n_volunteers = len(all_volunteers)

    print(f"\nFound volunteers: {all_volunteers}")
    print(f"Total volunteers: {n_volunteers}")

    # -------------------------------------------------------
    # SPLIT STRATEGY
    # -------------------------------------------------------
    if n_volunteers >= 10:
        n_folds = 5
        n_test = max(2, n_volunteers // 6)
        test_volunteers = all_volunteers[-n_test:]
        train_val_volunteers = all_volunteers[:-n_test]
    else:
        print(f"Warning: Only {n_volunteers} volunteers → using 3-fold CV")
        n_folds = min(3, n_volunteers)
        test_volunteers = []
        train_val_volunteers = all_volunteers

    print(f"Test set volunteers: {test_volunteers}")
    print(f"Train/Val volunteers: {train_val_volunteers}")

    # -------------------------------------------------------
    # ROUND-ROBIN FOLD ASSIGNMENT (ROBUST)
    # -------------------------------------------------------
    validation_groups = [[] for _ in range(n_folds)]
    for i, v in enumerate(train_val_volunteers):
        validation_groups[i % n_folds].append(v)

    print(f"Validation groups: {validation_groups}")

    # -------------------------------------------------------
    # BUILD SPLITS
    # -------------------------------------------------------
    splits = []
    for fold_idx, val_volunteers in enumerate(validation_groups):
        train_volunteers = [
            v for v in train_val_volunteers if v not in val_volunteers
        ]

        train_ids = []
        val_ids = []

        for f in all_files:
            if any(matches_volunteer(f, v) for v in train_volunteers):
                train_ids.append(f)
            elif any(matches_volunteer(f, v) for v in val_volunteers):
                val_ids.append(f)

        train_ids = sorted(set(train_ids))
        val_ids = sorted(set(val_ids))

        # HARD SAFETY CHECK
        assert len(val_ids) > 0, f"❌ Fold {fold_idx} has ZERO validation samples!"

        # Data leakage check
        train_found = {
            int(re.search(r'Volunteer_0*(\d+)_', f).group(1))
            for f in train_ids
        }
        val_found = {
            int(re.search(r'Volunteer_0*(\d+)_', f).group(1))
            for f in val_ids
        }

        overlap = train_found & val_found
        if overlap:
            raise RuntimeError(
                f"❌ DATA LEAKAGE in fold {fold_idx}: {sorted(overlap)}"
            )

        splits.append({"train": train_ids, "val": val_ids})

        print(
            f"✓ Fold {fold_idx}: "
            f"{len(train_ids)} train ({len(train_volunteers)} vols), "
            f"{len(val_ids)} val ({len(val_volunteers)} vols)"
        )

    # -------------------------------------------------------
    # SAVE SPLITS
    # -------------------------------------------------------
    output_path = os.path.join(
        "/home/sastocke/nnUNet/nnUNet_preprocessed",
        dataset_name,
        "splits_final.json"
    )
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    with open(output_path, "w") as f:
        json.dump(splits, f, indent=4)

    print(f"\nSplits saved to {output_path}")
    print("=" * 60)


dataset_names = [
    "Dataset101_HannumDirVsAveragesLV",
    "Dataset102_HannumSmartHealthandDirVsAvgs",
]

for dataset_name in dataset_names:
    specific_split_json_file(dataset_name)


Sample filenames: ['DirVsAvgHannum_Hannum_Volunteer_01_DiVO_06_10_slice_001', 'DirVsAvgHannum_Hannum_Volunteer_01_DiVO_06_10_slice_002', 'DirVsAvgHannum_Hannum_Volunteer_01_DiVO_06_10_slice_003']

Found volunteers: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
Total volunteers: 12
Test set volunteers: [11, 12]
Train/Val volunteers: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Validation groups: [[1, 6], [2, 7], [3, 8], [4, 9], [5, 10]]
✓ Fold 0: 137 train (8 vols), 36 val (2 vols)
✓ Fold 1: 139 train (8 vols), 34 val (2 vols)
✓ Fold 2: 137 train (8 vols), 36 val (2 vols)
✓ Fold 3: 141 train (8 vols), 32 val (2 vols)
✓ Fold 4: 138 train (8 vols), 35 val (2 vols)

Splits saved to /home/sastocke/nnUNet/nnUNet_preprocessed/Dataset101_HannumDirVsAveragesLV/splits_final.json
Sample filenames: ['DirVsAvgHannum_Hannum_Volunteer_01_DiVO_06_10_slice_001', 'DirVsAvgHannum_Hannum_Volunteer_01_DiVO_06_10_slice_002', 'DirVsAvgHannum_Hannum_Volunteer_01_DiVO_06_10_slice_003']

Found volunteers: [1, 2, 3, 4, 5, 6, 7, 8,